# TAIFEX Night Flow Collector v0.1.0

Set the target and persistence options below, then run all cells. Auto mode uses Taipei local time plus the checked-in verified TAIFEX calendar. The Drive folder ID is checked through Drive API before any upload. Do not run this Drive writer concurrently with a GitHub Actions collection.

In [ ]:
TARGET_DATE = "auto"  # or YYYY-MM-DD
SAVE_TO_DRIVE = True
DRY_RUN = False
DRIVE_FOLDER_ID = "1bHNCF59WGheGa5Tmo_GDK53T4fxU00oD"
REPO_URL = "https://github.com/hh4832/taifex-night-flow.git"


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_DIR = Path("/content/taifex-night-flow")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
%cd /content/taifex-night-flow
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(["uv", "venv", "--python", "3.11", ".venv"], cwd=REPO_DIR, check=True)
subprocess.run(["uv", "pip", "install", "--python", str(REPO_DIR / ".venv/bin/python"), "-e", "."], cwd=REPO_DIR, check=True)
PYTHON = str(REPO_DIR / ".venv/bin/python")


In [ ]:
from google.colab import drive, auth
from googleapiclient.discovery import build
import google.auth
import hashlib, json, shutil, tempfile
if SAVE_TO_DRIVE and not DRY_RUN:
    auth.authenticate_user()
    drive.mount("/content/drive")
    creds, _ = google.auth.default()
    service = build("drive", "v3", credentials=creds, cache_discovery=False)
    meta = service.files().get(fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType,parents,driveId").execute()
    assert meta["id"] == DRIVE_FOLDER_ID and meta["mimeType"] == "application/vnd.google-apps.folder", meta
    # Resolve the folder's actual My Drive path; fail if it is outside My Drive.
    names = [meta["name"]]
    parent_ids = meta.get("parents", [])
    while parent_ids:
        parent = service.files().get(fileId=parent_ids[0], fields="id,name,mimeType,parents").execute()
        if parent["id"] == "root": break
        names.append(parent["name"]); parent_ids = parent.get("parents", [])
    DRIVE_ROOT = Path("/content/drive/MyDrive").joinpath(*reversed(names))
    if not DRIVE_ROOT.is_dir(): raise RuntimeError(f"Verified Drive folder path is not mounted: {DRIVE_ROOT}")
    print(f"Verified Drive folder ID {meta['id']} at {DRIVE_ROOT}")


In [ ]:
DATA_DIR = Path("/content/taifex-night-flow-data")
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)
if SAVE_TO_DRIVE and not DRY_RUN:
    shutil.copytree(DRIVE_ROOT, DATA_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".state"))
args = ["--date", TARGET_DATE, "--data-dir", str(DATA_DIR), "--no-drive"]
if DRY_RUN: args.append("--dry-run")
run = subprocess.run([PYTHON, "-m", "taifex_night_flow.cli", *args], cwd=REPO_DIR, text=True, capture_output=True)
print(run.stdout)
if run.stderr: print(run.stderr)
print("collector_exit_code:", run.returncode)


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""): h.update(chunk)
    return h.hexdigest()

def is_successful_latest(path):
    try: return json.loads(path.read_text()).get("status") == "VALIDATED"
    except Exception: return False

if SAVE_TO_DRIVE and not DRY_RUN:
    # Immutable evidence is copied first. Existing immutable paths must match byte-for-byte.
    for src in DATA_DIR.rglob("*"):
        if not src.is_file() or ".state" in src.parts or src.name == "latest.json": continue
        rel = src.relative_to(DATA_DIR); dst = DRIVE_ROOT / rel; dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() and rel.as_posix() in {"manifest.json", "logs/fetch_events.csv", "logs/failures.csv"}:
            shutil.copy2(src, dst)  # Rebuildable indexes may be refreshed after immutable evidence.
        elif dst.exists():
            if sha256_file(src) != sha256_file(dst): raise RuntimeError(f"Immutable Drive collision: {rel}")
        else: shutil.copy2(src, dst)
    # Verify every source record is present before advancing the pointer.
    for src in DATA_DIR.rglob("*"):
        if src.is_file() and ".state" not in src.parts and src.name != "latest.json":
            dst = DRIVE_ROOT / src.relative_to(DATA_DIR)
            if not dst.exists() or sha256_file(src) != sha256_file(dst): raise RuntimeError(f"Drive verification failed: {src.relative_to(DATA_DIR)}")
    latest = DATA_DIR / "latest.json"
    if is_successful_latest(latest):
        temp = DRIVE_ROOT / ".latest.json.tmp"
        shutil.copy2(latest, temp); os.replace(temp, DRIVE_ROOT / "latest.json")
        verified = json.loads((DRIVE_ROOT / "latest.json").read_text())
        assert verified["trade_date"] == json.loads(latest.read_text())["trade_date"]
    print("Drive synchronization complete:", DRIVE_ROOT)
elif SAVE_TO_DRIVE and DRY_RUN:
    print("DRY_RUN enabled: no Drive or persistent local data was changed.")
elif not SAVE_TO_DRIVE:
    print("Local output:", DATA_DIR)
